In [0]:
%sql
-- Q01: Overall KPI summary: net revenue, orders, active customers, average order value
-- Business interpretation: This provides a top-line view of total business performance across all periods.
--   The average order value indicates the typical transaction size and helps gauge customer spending behavior.
-- Management action: Use these KPIs as baseline targets for quarterly performance reviews and set growth targets for each metric.
SELECT
    SUM(revenue) AS net_revenue,
    COUNT(DISTINCT order_key) AS total_orders,
    COUNT(DISTINCT customer_key) AS active_customers,
    ROUND(SUM(revenue) / COUNT(DISTINCT order_key), 2) AS avg_order_value
FROM tpch_dev.gold.fact_sales

In [0]:
%sql
-- Q02: Monthly revenue trend across all years
-- Business interpretation: The monthly trend reveals seasonal patterns and growth trajectories across the full time horizon.
--   Consistent monthly revenue indicates stable demand, while significant fluctuations may signal seasonality or market shifts.
-- Management action: Use seasonal patterns to optimize inventory, staffing, and marketing spend throughout the year.
SELECT
    DATE_TRUNC('month', order_date) AS month,
    SUM(revenue) AS monthly_revenue
FROM tpch_dev.gold.fact_sales
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY month

In [0]:
%sql
-- Q03: Annual revenue with year-over-year change
-- Business interpretation: Year-over-year revenue change shows whether the business is growing or declining.
--   Consistent positive YoY growth indicates healthy business expansion, while negative growth warrants investigation.
-- Management action: If YoY growth is declining, conduct a root-cause analysis covering pricing, competition, and customer retention.
SELECT
    YEAR(order_date) AS sales_year,
    SUM(revenue) AS annual_revenue,
    LAG(SUM(revenue)) OVER (ORDER BY YEAR(order_date)) AS prev_year_revenue,
    ROUND(
        (SUM(revenue) - LAG(SUM(revenue)) OVER (ORDER BY YEAR(order_date))) /
        LAG(SUM(revenue)) OVER (ORDER BY YEAR(order_date)) * 100, 2
    ) AS yoy_pct_change
FROM tpch_dev.gold.fact_sales
GROUP BY YEAR(order_date)
ORDER BY sales_year

In [0]:
%sql
-- Q04: Revenue by customer market segment
-- Business interpretation: Revenue by market segment identifies which customer segments drive the most revenue.
--   Segments with high revenue but low customer counts represent concentrated risk, while balanced segments indicate diversified income.
-- Management action: Allocate marketing budgets proportionally to high-revenue segments while developing growth plans for underperforming segments.
SELECT
    dc.market_segment,
    ROUND(SUM(fs.revenue), 2) AS net_revenue,
    COUNT(DISTINCT fs.order_key) AS order_count,
    COUNT(DISTINCT fs.customer_key) AS customer_count
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
GROUP BY dc.market_segment
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q05: Top 10 customers by net revenue
-- Business interpretation: The top 10 customers by net revenue reveal customer concentration risk.
--   If a few customers account for a disproportionate share of revenue, the business is vulnerable to losing key accounts.
-- Management action: Develop key account management programs for top customers to strengthen relationships and reduce churn risk.
SELECT
    dc.customer_key,
    dc.customer_name,
    dc.market_segment,
    dc.nation_name,
    ROUND(SUM(fs.revenue), 2) AS net_revenue,
    COUNT(DISTINCT fs.order_key) AS order_count
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
GROUP BY dc.customer_key, dc.customer_name, dc.market_segment, dc.nation_name
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Q06: Top 10 customers as a share of total revenue
-- Business interpretation: This quantifies customer concentration by showing what percentage of total revenue comes from the top 10 customers.
--   A high percentage indicates significant dependency on a small number of accounts.
-- Management action: If the top 10 share exceeds 20%, prioritize customer acquisition campaigns to diversify the revenue base.
WITH top_customers AS (
    SELECT
        dc.customer_key,
        dc.customer_name,
        SUM(fs.revenue) AS customer_revenue
    FROM tpch_dev.gold.fact_sales fs
    JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
    GROUP BY dc.customer_key, dc.customer_name
    ORDER BY customer_revenue DESC
    LIMIT 10
),
total_revenue AS (
    SELECT SUM(revenue) AS total FROM tpch_dev.gold.fact_sales
)
SELECT
    tc.customer_key,
    tc.customer_name,
    ROUND(tc.customer_revenue, 2) AS customer_revenue,
    ROUND(tr.total, 2) AS total_revenue,
    ROUND(tc.customer_revenue / tr.total * 100, 4) AS revenue_share_pct
FROM top_customers tc
CROSS JOIN total_revenue tr
ORDER BY tc.customer_revenue DESC

In [0]:
%sql
-- Q07: Top 10 products by net revenue
-- Business interpretation: The top 10 products by net revenue identify the best-performing products in the portfolio.
--   These products are the primary revenue drivers and should be protected from supply disruptions.
-- Management action: Ensure adequate inventory levels for top products and consider promotional support to sustain their performance.
SELECT
    dp.product_key,
    dp.product_name,
    dp.brand,
    dp.product_type,
    ROUND(SUM(fs.revenue), 2) AS net_revenue,
    ROUND(SUM(fs.quantity), 2) AS total_quantity
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_product dp ON fs.product_key = dp.product_key
GROUP BY dp.product_key, dp.product_name, dp.brand, dp.product_type
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Q08: Bottom 10 products by net revenue
-- Business interpretation: The bottom 10 products by net revenue highlight underperforming items in the catalog.
--   These products may suffer from low demand, high discounts, or poor market fit.
-- Management action: Review bottom products for potential discontinuation or repositioning and reallocate resources to higher-performing items.
SELECT
    dp.product_key,
    dp.product_name,
    dp.brand,
    dp.product_type,
    ROUND(SUM(fs.revenue), 2) AS net_revenue,
    ROUND(SUM(fs.quantity), 2) AS total_quantity
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_product dp ON fs.product_key = dp.product_key
GROUP BY dp.product_key, dp.product_name, dp.brand, dp.product_type
ORDER BY net_revenue ASC
LIMIT 10

In [0]:
%sql
-- Q09: Revenue by product type
-- Business interpretation: Revenue by product type shows which product categories generate the most income.
--   High-revenue categories should receive continued investment, while low-revenue categories may need strategic review.
-- Management action: Invest in product development and marketing for high-revenue types while evaluating the strategic fit of low-revenue categories.
SELECT
    dp.product_type,
    ROUND(SUM(fs.revenue), 2) AS net_revenue,
    COUNT(DISTINCT dp.product_key) AS product_count,
    ROUND(SUM(fs.quantity), 2) AS total_quantity
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_product dp ON fs.product_key = dp.product_key
GROUP BY dp.product_type
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q10: Top 10 suppliers by net revenue
-- Business interpretation: The top 10 suppliers by net revenue identify key supply chain partners.
--   These suppliers are critical to revenue generation and should be nurtured through strategic partnerships.
-- Management action: Establish strategic supplier agreements with top partners to secure favorable terms and ensure supply continuity.
SELECT
    ds.supplier_key,
    ds.supplier_name,
    ds.nation_name,
    ds.region_name,
    ROUND(SUM(fs.revenue), 2) AS net_revenue,
    COUNT(DISTINCT fs.order_key) AS order_count
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_supplier ds ON fs.supplier_key = ds.supplier_key
GROUP BY ds.supplier_key, ds.supplier_name, ds.nation_name, ds.region_name
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Q11: Revenue share of the top 10 suppliers
-- Business interpretation: This quantifies supplier concentration risk by showing what percentage of total revenue comes from the top 10 suppliers.
--   A high percentage suggests over-reliance on a small number of suppliers, increasing supply chain vulnerability.
-- Management action: If the top 10 supplier share exceeds 30%, develop alternative supplier relationships to mitigate supply chain risk.
WITH top_suppliers AS (
    SELECT
        ds.supplier_key,
        ds.supplier_name,
        SUM(fs.revenue) AS supplier_revenue
    FROM tpch_dev.gold.fact_sales fs
    JOIN tpch_dev.gold.dim_supplier ds ON fs.supplier_key = ds.supplier_key
    GROUP BY ds.supplier_key, ds.supplier_name
    ORDER BY supplier_revenue DESC
    LIMIT 10
),
total_revenue AS (
    SELECT SUM(revenue) AS total FROM tpch_dev.gold.fact_sales
)
SELECT
    tc.supplier_key,
    tc.supplier_name,
    ROUND(tc.supplier_revenue, 2) AS supplier_revenue,
    ROUND(tr.total, 2) AS total_revenue,
    ROUND(tc.supplier_revenue / tr.total * 100, 4) AS revenue_share_pct
FROM top_suppliers tc
CROSS JOIN total_revenue tr
ORDER BY tc.supplier_revenue DESC

In [0]:
%sql
-- Q12: Revenue by customer region
-- Business interpretation: Revenue by customer region shows geographic revenue distribution across broad regions.
--   Regions with high revenue represent strong markets, while low-revenue regions may benefit from targeted marketing.
-- Management action: Invest in high-revenue regions through localized marketing and consider market entry studies for underperforming regions.
SELECT
    dc.region_name,
    ROUND(SUM(fs.revenue), 2) AS net_revenue,
    COUNT(DISTINCT fs.customer_key) AS customer_count,
    COUNT(DISTINCT fs.order_key) AS order_count
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
GROUP BY dc.region_name
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q13: Revenue by customer nation (all nations)
-- Business interpretation: Revenue by customer nation provides a granular geographic revenue breakdown.
--   This enables country-specific strategy adjustments and identifies emerging or declining markets.
-- Management action: Tailor country-specific pricing and marketing strategies based on revenue contribution and growth potential.
SELECT
    dc.nation_name,
    dc.region_name,
    ROUND(SUM(fs.revenue), 2) AS net_revenue,
    COUNT(DISTINCT fs.customer_key) AS customer_count,
    COUNT(DISTINCT fs.order_key) AS order_count
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
GROUP BY dc.nation_name, dc.region_name
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q14: Revenue and volume split by return flag
-- Business interpretation: The return flag analysis shows the revenue and volume split between returned and non-returned items.
--   A high return rate may indicate quality issues or customer satisfaction problems that need addressing.
-- Management action: If the return rate exceeds 5%, initiate a quality review and customer feedback analysis to address root causes.
SELECT
    fs.return_flag,
    ROUND(SUM(fs.revenue), 2) AS net_revenue,
    ROUND(SUM(fs.quantity), 2) AS total_quantity,
    COUNT(*) AS line_count,
    ROUND(SUM(fs.revenue) / (SELECT SUM(revenue) FROM tpch_dev.gold.fact_sales) * 100, 4) AS revenue_share_pct
FROM tpch_dev.gold.fact_sales fs
GROUP BY fs.return_flag
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q15: Average discount by market segment
-- Business interpretation: Average discount by market segment reveals which segments receive the deepest discounts.
--   High discount rates may erode margins, suggesting the need to review pricing strategies for those segments.
-- Management action: Review discount policies for high-discount segments to protect margins while maintaining customer loyalty.
SELECT
    dc.market_segment,
    ROUND(AVG(fs.discount_rate) * 100, 4) AS avg_discount_pct,
    ROUND(SUM(fs.revenue), 2) AS net_revenue,
    ROUND(SUM(fs.discount_amount), 2) AS total_discount_amount,
    COUNT(*) AS line_count
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
GROUP BY dc.market_segment
ORDER BY avg_discount_pct DESC

In [0]:
%sql
-- Q16: Average ship-to-receipt days by shipping mode
-- Business interpretation: Average ship-to-receipt days by shipping mode identifies the most and least efficient shipping methods.
--   Slower shipping modes may need process improvements or replacement to enhance customer satisfaction.
-- Management action: Prioritize faster shipping modes for high-value orders and negotiate better SLAs with shipping partners for underperforming modes.
SELECT
    fs.ship_mode,
    ROUND(AVG(DATEDIFF(fs.receipt_date, fs.ship_date)), 2) AS avg_ship_to_receipt_days,
    ROUND(AVG(DATEDIFF(fs.ship_date, fs.order_date)), 2) AS avg_order_to_ship_days,
    COUNT(*) AS line_count
FROM tpch_dev.gold.fact_sales fs
GROUP BY fs.ship_mode
ORDER BY avg_ship_to_receipt_days DESC

In [0]:
%sql
-- Q17: Order count and revenue by order priority
-- Business interpretation: Order count and revenue by order priority shows how different priority levels contribute to business.
--   Understanding the revenue impact of each priority level helps optimize order processing workflows.
-- Management action: Balance order processing resources across priority levels to ensure high-priority orders are fulfilled without delay.
SELECT
    fs.order_priority,
    COUNT(DISTINCT fs.order_key) AS order_count,
    ROUND(SUM(fs.revenue), 2) AS net_revenue,
    ROUND(SUM(fs.revenue) / COUNT(DISTINCT fs.order_key), 2) AS avg_order_value
FROM tpch_dev.gold.fact_sales fs
GROUP BY fs.order_priority
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q18: Monthly breakdown of order status
-- Business interpretation: The monthly breakdown of order status reveals how order fulfillment evolves over time.
--   Tracking status changes monthly helps identify bottlenecks in the order lifecycle and plan capacity accordingly.
-- Management action: Investigate months with high pending or backordered statuses to identify and resolve fulfillment bottlenecks.
SELECT
    DATE_TRUNC('month', fs.order_date) AS month,
    fs.order_status,
    COUNT(DISTINCT fs.order_key) AS order_count,
    ROUND(SUM(fs.revenue), 2) AS net_revenue
FROM tpch_dev.gold.fact_sales fs
GROUP BY DATE_TRUNC('month', fs.order_date), fs.order_status
ORDER BY month, fs.order_status

# Final Business Challenge: Business Performance Summary

## "How is our business performing, and where should management focus its attention?"

**Overall Performance (Q01):** Across the full 1992-1998 period, the business generated **$1,089,835,179,247** (~$1.09T) in net revenue from **7,500,000 orders** placed by **499,989 active customers**, with an average order value of **$145,311**.

---

## Three Key Findings (Data-Supported)

### Finding 1: Revenue Plateau (1992-1997) with 1998 Decline
**Supporting Query:** Q03 — Annual revenue with YoY change
**Metric:** 1992-1997 annual revenue averaged ~$165.5B/year with YoY changes between -0.26% and +0.57%. 1998 revenue dropped to $96.97B (-41.32% YoY).

**Interpretation:** Revenue was remarkably stable across six full years (1992-1997), with no meaningful growth or decline. The 1998 figure of $97.0B is substantially lower, but this likely reflects partial-year data in the dataset rather than a confirmed business contraction.

**Recommended Management Action:** Treat 1992-1997 as the baseline performance level (~$165B/year). Verify whether 1998 data covers the full calendar year before drawing conclusions about a downturn. If confirmed partial, focus on year-over-year stability as the key narrative for stakeholders.

---

### Finding 2: Extreme Revenue Diversification — No Concentration Risk
**Supporting Query:** Q06 (Top 10 customers share) and Q11 (Top 10 suppliers share)
**Metric:** Top 10 customers contribute $66.9M (0.0061% of total revenue). Top 10 suppliers contribute $288.9M (0.0265% of total revenue).

**Interpretation:** Revenue is distributed across an extremely wide base of ~500K customers and ~100K suppliers. No single customer or supplier represents material concentration risk. The business model is highly diversified.

**Recommended Management Action:** No urgent action needed for concentration risk. However, the absence of large key accounts suggests an opportunity to develop enterprise-level relationships that could deepen engagement and increase per-customer revenue beyond the current ~$2.18M average.

---

### Finding 3: Uniform Performance Across All Business Dimensions
**Supporting Query:** Q04 (market segment), Q12 (region), Q17 (order priority)
**Metric:**
- Market segments range from $216.6B (HOUSEHOLD) to $219.0B (FURNITURE) — spread of 1.1%
- Regions range from $216.7B (AFRICA) to $219.0B (EUROPE) — spread of 1.0%
- Order priorities each have ~1.5M orders and ~$218B revenue

**Interpretation:** No single segment, region, or order priority significantly outperforms or underperforms. The business is perfectly balanced across all dimensions, suggesting either highly effective market distribution or a lack of differentiated growth strategy.

**Recommended Management Action:** Investigate whether the uniform distribution reflects a deliberate diversification strategy or an absence of targeted growth initiatives. Pilot concentrated growth efforts in the highest-revenue segment (FURNITURE, $219.0B) and region (EUROPE, $219.0B) to test for scalable upside.

---

## Two Opportunities for Growth or Operational Improvement

### Opportunity 1: Return Rate Reduction
**Supporting Query:** Q14 — Revenue and volume split by return flag
**Metric:** 49.37% of line items (14.81M of 30.0M) carry a return flag of R (returned: 24.69%) or A (returned-accepted: 24.68%), representing $538.1B in associated revenue. Non-returned items (N) account for 50.63% ($551.7B).

**Interpretation:** Nearly half of all order lines are associated with returns. Even a modest reduction in the return rate could recover significant revenue and reduce reverse logistics costs. The near-even split between R and A flags suggests a substantial volume of items that are returned and not accepted back.

**Recommended Management Action:** Analyze return reasons by product type (Q09) and supplier (Q10) to identify systemic quality issues. Set a target to reduce the R-flag rate by 5 percentage points (from ~24.7% to ~19.7%) over the next fiscal year through quality improvements and supplier scorecards.

---

### Opportunity 2: Shipping Mode Consolidation
**Supporting Query:** Q16 — Average ship-to-receipt days by shipping mode
**Metric:** All 7 shipping modes have nearly identical average ship-to-receipt times (15.50-15.51 days). Each mode handles ~4.28M line items with ~$155.7B revenue.

**Interpretation:** No shipping mode offers a measurable speed advantage. The uniformity in delivery times and volume across 7 modes suggests an opportunity to consolidate to fewer carriers, simplifying logistics and improving negotiating leverage.

**Recommended Management Action:** Pilot consolidating from 7 shipping modes to 3-4 modes (e.g., AIR, TRUCK, RAIL) and measure cost savings vs. any impact on delivery performance. Negotiate volume-based discounts with the remaining carriers, leveraging the ~$155B per-mode spend.

---

## Two Risks Requiring Further Investigation

### Risk 1: 1998 Revenue Anomaly
**Supporting Query:** Q03 — Annual revenue with YoY change
**Metric:** 1998 revenue = $96.97B, a 41.32% decline from 1997's $165.25B.

**Interpretation:** The sharp decline could be caused by incomplete data (partial calendar year) or a genuine business downturn. Without confirming data completeness, any conclusion about a real decline would be speculative. This is flagged as a risk because if the decline is real, it would represent the most significant business contraction in the dataset.

**Recommended Management Action:** Verify whether 1998 data covers the full calendar year by checking the latest order_date in the fact table. If partial, annualize the figure for comparison. If complete, conduct an immediate root-cause analysis covering order volume trends (Q18), customer churn, and regional performance shifts.

---

### Risk 2: Uniform Discount Rate Suggests Limited Pricing Flexibility
**Supporting Query:** Q15 — Average discount by market segment
**Metric:** All 5 market segments have average discount rates between 4.9975% (AUTOMOBILE) and 5.0008% (FURNITURE) — a spread of only 0.0033 percentage points.

**Interpretation:** The extreme uniformity of discount rates across all segments is unusual and may indicate a rigid, centrally-mandated pricing policy rather than market-responsive pricing. This limits the ability to use discounts as a competitive tool in specific segments and may leave margin on the table in segments where lower discounts would be acceptable.

**Recommended Management Action:** Review the discount policy to determine whether the uniformity is intentional. Test segment-specific discount strategies in a controlled pilot — for example, reducing discounts in FURNITURE (the highest-revenue segment at $219.0B) by 1 percentage point could recover ~$2.2B in margin if volume holds.

---

*Note: This analysis is based on the TPC-H synthetic benchmark dataset. All findings reflect patterns in synthetic data and should not be interpreted as evidence of real-world business causation. Assumptions and hypotheses are clearly labeled and separated from data-supported findings.*

In [0]:
%sql

-- Q19: Product revenue ranking within each market segment
-- Business interpretation: Ranking products within each market segment identifies the top-performing products for each customer segment.
--   Products ranked highly within a segment represent strong fit and should be prioritized for that segment's customers.
-- Management action: Tailor product recommendations and promotions by market segment based on these rankings to maximize cross-sell opportunities.
WITH product_segment_revenue AS (
    SELECT
        dc.market_segment,
        dp.product_key,
        dp.product_name,
        dp.brand,
        dp.product_type,
        ROUND(SUM(fs.revenue), 2) AS net_revenue
    FROM tpch_dev.gold.fact_sales fs
    JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
    JOIN tpch_dev.gold.dim_product dp ON fs.product_key = dp.product_key
    GROUP BY dc.market_segment, dp.product_key, dp.product_name, dp.brand, dp.product_type
)
SELECT
    market_segment,
    product_key,
    product_name,
    brand,
    product_type,
    net_revenue,
    RANK() OVER (PARTITION BY market_segment ORDER BY net_revenue DESC) AS revenue_rank
FROM product_segment_revenue
ORDER BY market_segment, revenue_rank

In [0]:
%sql

-- Q20: Revenue comparison of customer geography vs supplier geography by region
-- Business interpretation: Comparing revenue by customer region against revenue by supplier region reveals supply-demand geographic alignment.
--   When customer and supplier regions overlap on high revenue, the supply chain is well-aligned with demand.
--   Mismatches — high customer revenue in a region supplied primarily by another region — indicate cross-regional dependency and logistics cost.
-- Management action: Where customer and supplier regions are misaligned, evaluate local sourcing options to reduce shipping costs and lead times.
WITH customer_region_revenue AS (
    SELECT
        dc.region_name AS customer_region,
        ROUND(SUM(fs.revenue), 2) AS customer_region_revenue
    FROM tpch_dev.gold.fact_sales fs
    JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
    GROUP BY dc.region_name
),
supplier_region_revenue AS (
    SELECT
        ds.region_name AS supplier_region,
        ROUND(SUM(fs.revenue), 2) AS supplier_region_revenue
    FROM tpch_dev.gold.fact_sales fs
    JOIN tpch_dev.gold.dim_supplier ds ON fs.supplier_key = ds.supplier_key
    GROUP BY ds.region_name
)
SELECT
    COALESCE(cr.customer_region, sr.supplier_region) AS region_name,
    ROUND(COALESCE(cr.customer_region_revenue, 0), 2) AS customer_region_revenue,
    ROUND(COALESCE(sr.supplier_region_revenue, 0), 2) AS supplier_region_revenue,
    ROUND(COALESCE(cr.customer_region_revenue, 0) - COALESCE(sr.supplier_region_revenue, 0), 2) AS revenue_difference,
    ROUND(
        COALESCE(cr.customer_region_revenue, 0) /
        NULLIF((SELECT SUM(revenue) FROM tpch_dev.gold.fact_sales), 0) * 100, 4
    ) AS customer_region_share_pct,
    ROUND(
        COALESCE(sr.supplier_region_revenue, 0) /
        NULLIF((SELECT SUM(revenue) FROM tpch_dev.gold.fact_sales), 0) * 100, 4
    ) AS supplier_region_share_pct
FROM customer_region_revenue cr
FULL OUTER JOIN supplier_region_revenue sr
    ON cr.customer_region = sr.supplier_region
ORDER BY region_name